# Table of Contents
- 1. Correlation Matrix Heatmaps (Pearson, Spearman, Cramér's V)
- 2. Variance Inflation Factor (VIF) for Collinearity
- 3. Partial Correlation Analysis
- 4. Non-Linear Association Detection (Maximal Information Coefficient)

In [2]:
# Global Imports and Runtime Visual Configuration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Visual styling settings
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 10

print("Global libraries for correlation and collinearity discovery loaded successfully.")
np.random.seed(42)

Global libraries for correlation and collinearity discovery loaded successfully.


---
## 1. Correlation Matrix Heatmaps (Pearson, Spearman, Cramér's V)
### 1. Definition
Correlation matrix heatmaps quantify and visualize 
- Linear relationships (Pearson), 
- Monotonic associations (Spearman rank), or 
- Categorical associations (Cramér's V) across feature pairs in a tabular matrix.

### 2. Math Formula
Pearson correlation coefficient $r_{xy}$ between vectors $x$ and $y$ is:
$$r_{xy} = \frac{\sum_{i=1}^N (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i=1}^N (x_i - \bar{x})^2 \sum_{i=1}^N (y_i - \bar{y})^2}}$$

### 3. Notation Pronunciation
- $r_{xy}$: "Pearson correlation coefficient between x and y"
- $\rho$: "Spearman rank correlation"
- `sns.heatmap()`: "Seaborn matrix heatmap visualization"

### 4. Naive Example
Plotting a correlation heatmap of customer dataset features to check if annual income and credit score are highly correlated before building a loan approval model.

### 5. Uses
- Detecting redundant features and multicollinearity risks.
- Understanding linear and rank-based dependencies between variables.
- Summarizing dataset association structures visually.

In [ ]:
import numpy as np
import pandas as pd
hours = np.array([1,2,3,4,5,6,7,8])
scores = 10 * hours + 5 + np.random.normal(0,3, size=len(hours))
df_linear = pd.DataFrame({'Hours Studided':hours, 'Exam score':scores})

linear_corr = df_linear['Hours Studided'].corr(df_linear['Exam score'], method='pearson')
print(f"Linear Correlation (Pearson): {linear_corr:.2f}")
print(df_linear)

Linear Correlation (Pearson): 1.00



## 2. Monotonic Relationship

- A **monotonic relationship** means that as one variable increases, the other variable always moves in the same direction (either it always increases or it always decreases), but not necessarily at a constant rate.
- The relationship can curve, flatten out, or accelerate (like exponential or logarithmic growth), but it never reverses direction.
- **Spearman Rank Correlation** is used to measure this relationship. It looks at the rank (order) of the data rather than the exact values.

### Example: Advertising Budget vs. Product Sales (Diminishing Returns)

- Increasing the advertising budget from $2k to $4k may cause a massive jump in sales.
- Increasing the advertising budget from $11k to $13k may cause a smaller jump.
- Sales always increase as advertising increases, but the rate of growth slows down, creating a curved (non-linear but monotonic) pattern.

## 3. Spearman Rank Correlation (ρ or rₛ)

- **Value Range:** -1 to +1 (Just like Pearson correlation).
- **Sign Meaning:** The sign (+ or -) tells you the direction of the relationship, based on ranks (ordering) rather than raw values.

  - **+1:** Perfectly monotonic increasing relationship (as X increases, Y always increases).
  - **0:** No monotonic relationship.
  - **-1:** Perfectly monotonic decreasing relationship (as X increases, Y always decreases).

### Rule of Thumb for Strength

- Values closer to **+1** indicate a strong positive monotonic relationship.
- Values closer to **-1** indicate a strong negative monotonic relationship.
- Values closer to **0** indicate a weak or no monotonic relationship.


In [ ]:
import numpy as np
import pandas as pd
np.random.seed(42)
# 2. Monotonic Example: Ad Spend vs Sales (Diminishing returns curve)
ad_spend = np.array([1, 2, 5, 10, 20, 50, 100])
# Sales grow logarithmically—always going up, but at a decreasing rate
sales = np.log(ad_spend) * 50 + 100

df_monotonic = pd.DataFrame({'Ad_Spend': ad_spend, 'Sales': sales})
# Calculate Spearman correlation (measures monotonic relationships)
monotonic_corr = df_monotonic['Ad_Spend'].corr(df_monotonic['Sales'], method='spearman')
print(f"\nMonotonic Correlation (Spearman): {monotonic_corr:.2f}")
print(df_monotonic)


## 3. Categorical Association

- A **categorical association** looks at whether two non-numeric, group-based variables are related to each other. Instead of numbers, your data consists of labels or categories.
- **Cramér's V** is typically used here. It is calculated from a contingency (cross-tabulation) table and ranges from 0 (no association) to 1 (perfect association).
- **Novice Example:** Pet Ownership (Dog, Cat, None) vs. Preferred Housing Type (Apartment, House). We want to see if owning a specific type of pet is associated with living in a certain type of home.

### 3.1 Cramér's V

- **Value Range:** 0 to 1 (It never goes negative because categories like "Dog" vs. "Cat" don't have a directional "positive" or "negative" value).

- **Meaning:**
  - **0:** No association (knowing one category tells you nothing about the other category).
  - **1:** Perfect association (knowing one category tells you exactly what the other category is).

- **Rule of Thumb for Strength** (depends slightly on the table size, but generally):
  - **0 to 0.10:** Negligible association
  - **0.10 to 0.20:** Weak association
  - **0.20 to 0.40:** Moderate association
  - **Above 0.40:** Strong association


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
np.random.seed(42)
# 3. Categorical Example: Pet Preference vs Housing Type
data = {
    'Pet': ['Dog', 'Cat', 'Dog', 'None', 'Cat', 'Dog', 'None', 'Cat', 'Dog', 'None'],
    'Housing': ['House', 'Apartment', 'House', 'Apartment', 'Apartment', 'House', 'Apartment', 'House', 'House', 'Apartment']
}
df_cat = pd.DataFrame(data)
# print(df_cat)
# Create a contingency table (cross-tabulation)
contingency_table = pd.crosstab(df_cat['Pet'], df_cat['Housing'])
chi2, p, dof, ex = chi2_contingency(contingency_table)
print("chi2 :  Chi-Square Statistics\n",chi2)
print("p : p-value\n",p)
print("dof : degree of freedom\n",dof)
print("ex : Expected Frequencies / Expected Matrix\n",ex)
n = contingency_table.sum().sum()
print(n)
min_dim = min(contingency_table.shape) - 1
print("min_dim\n",min_dim)
cramers_v = np.sqrt(chi2 / (n * min_dim))

print(f"\nCategorical Association (Cramér's V): {cramers_v:.2f}")
print("\nContingency Table:")
print(contingency_table)


In [ ]:
np.random.seed(42)
df_corr = pd.DataFrame({
    'feature_1': np.random.rand(100),
    'feature_2': np.random.rand(100) * 2
})
df_corr['feature_3'] = df_corr['feature_1'] * 0.8 + np.random.normal(0, 0.1, 100)
corr_matrix = df_corr.corr(method='pearson')
print("Pearson Correlation Matrix:\n", corr_matrix)

# Visualization
plt.figure(figsize=(6, 4))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Matrix Heatmap')
plt.show()


## 2. Variance Inflation Factor (VIF) for Collinearity

### 1. Definition

- **Variance Inflation Factor (VIF)** detects and measures the severity of multicollinearity in regression models.
- It quantifies how much the variance of an estimated regression coefficient is inflated due to correlation between independent variables.
- High VIF indicates that a feature is highly correlated with other independent features.

### 2. Mathematical Formula

For feature \(X_j\), VIF is computed using the coefficient of determination (\(R_j^2\)) obtained by regressing feature \(X_j\) against all other independent features.

\[
VIF_j = \frac{1}{1-R_j^2}
\]

### 3. Notation Pronunciation

- \(VIF_j\): "Variance Inflation Factor for feature j"
- \(R_j^2\): "R-squared coefficient of determination for feature j"
- \(X_j\): "Feature j"
- \(1-R_j^2\): "One minus R-squared"

### 4. Naive Example

Consider a housing price dataset containing the following features:

- Square Footage
- Number of Rooms
- Number of Bathrooms
- House Age

Suppose we calculate VIF and get:

| Feature | VIF |
|---|---:|
| Square Footage | 12.5 |
| Number of Rooms | 11.2 |
| Number of Bathrooms | 2.5 |
| House Age | 1.8 |

**Interpretation:**

- Square Footage has a VIF of 12.5, indicating high multicollinearity.
- Number of Rooms has a VIF of 11.2, also indicating high multicollinearity.
- Number of Bathrooms has a VIF of 2.5, which is generally acceptable.
- House Age has a VIF of 1.8, indicating low multicollinearity.

Square Footage and Number of Rooms may contain overlapping information because larger houses generally have more rooms.

### 5. VIF Interpretation (Rule of Thumb)

| VIF Value | Interpretation |
|---|---|
| 1 | No multicollinearity |
| 1–5 | Low to moderate multicollinearity |
| 5–10 | High multicollinearity |
| Above 10 | Severe multicollinearity |

*Note: These are common guidelines, not strict statistical boundaries.*

### 6. Uses

- Identifying redundant predictor variables in multiple linear regression.
- Deciding which features to drop or regularize to stabilize coefficient estimates.
- Detecting multicollinearity between independent variables.
- Improving model interpretability and the reliability of coefficient estimates.

### 7. Quick Revision

- VIF measures multicollinearity.
- VIF = 1 means no linear correlation with other predictors.
- Higher VIF indicates greater multicollinearity.
- VIF above 5 deserves investigation.
- VIF above 10 is commonly considered severe.


In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_vif = pd.DataFrame({
    'x1': [1, 2, 3, 4, 5],
    'x2': [2.1, 3.9, 6.2, 7.9, 10.1], # Perfectly collinear with x1
    'x3': [1, 0, 3, 2, 5]
})
vif_data = pd.DataFrame()
vif_data["Feature"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]


## 3. Partial Correlation Analysis

### 1. Definition

- **Partial correlation** measures the degree of association between two variables while removing the confounding linear effect of one or more controlling variables.
- It helps us understand the relationship between two variables independently of other influencing variables.
- Unlike simple correlation, partial correlation considers the effect of additional variables.

### 2. Mathematical Formula

The partial correlation between \(X\) and \(Y\), controlling for \(Z\), is derived from their bivariate correlation coefficients:

\[
r_{XY.Z} =
\frac{r_{XY} - r_{XZ}r_{YZ}}
{\sqrt{(1-r_{XZ}^{2})(1-r_{YZ}^{2})}}
\]

### 3. Notation Pronunciation

- \(r_{XY.Z}\): "Partial correlation of X and Y controlling for Z"
- \(r_{XY}\): "Correlation between X and Y"
- \(r_{XZ}\): "Correlation between X and Z"
- \(r_{YZ}\): "Correlation between Y and Z"

### 4. Naive Example

**Ice Cream Sales vs. Shark Attacks**

Suppose we observe that ice cream sales increase whenever shark attacks increase.

At first, it may appear that these two variables are related.

However, ambient temperature is a confounding variable:

- During hot weather, people buy more ice cream.
- During hot weather, more people visit beaches and swim in the ocean.
- More people swimming can lead to more shark encounters.

By controlling for ambient temperature, we can measure the relationship between ice cream sales and shark attacks without the linear effect of temperature.

### 5. Uses

- Isolating relationships between variables by removing the linear effects of confounding variables.
- Advanced causal and exploratory statistical analysis.
- Refining feature selection in multivariate models.
- Understanding whether an observed correlation remains after controlling for other variables.

### 6. Quick Revision

- Partial correlation measures the relationship between two variables while controlling for other variables.
- It removes the linear influence of controlling variables.
- It ranges from -1 to +1.
- A value close to +1 indicates a strong positive relationship.
- A value close to -1 indicates a strong negative relationship.
- A value close to 0 indicates little or no remaining linear association.


In [ ]:
np.random.seed(42)
z = np.random.randn(100)
x = z + np.random.randn(100) * 0.5
y = z + np.random.randn(100) * 0.5
# Simulated partial correlation logic
df_part = pd.DataFrame({'x': x, 'y': y, 'z': z})
corr_xy = df_part[['x', 'y']].corr().iloc[0,1]
print(f"Raw Bivariate Correlation (X, Y): {corr_xy:.3f}")


## 4. Non-Linear Association Detection (Maximal Information Coefficient)

### 1. Definition

- **Maximal Information Coefficient (MIC)** captures both linear and complex non-linear associations (such as exponential, periodic, or sinusoidal patterns) between variables.
- It can detect relationships that traditional Pearson correlation may fail to identify.
- MIC is based on information theory and measures how much information one variable provides about another.

### 2. Mathematical Formula

MIC is based on **Mutual Information (MI)** over different grid partitions of the data.

\[
MIC(X,Y) =
\max_{|G|<B(n)}
\frac{I(X,Y|G)}
{\log(\min\{x,y\})}
\]

Where:

- \(I(X,Y|G)\): Mutual information between X and Y for a particular grid partition G.
- \(G\): A grid partition of the data.
- \(x\): Number of columns in the grid.
- \(y\): Number of rows in the grid.
- \(B(n)\): Maximum grid complexity, which depends on the number of observations \(n\).
- \(\max\): Selects the maximum normalized mutual information across allowed grids.

**Note:** MIC searches across different grid sizes and partitions to find the strongest normalized association.

### 3. Notation Pronunciation

- \(MIC(X,Y)\): "Maximal Information Coefficient between X and Y"
- \(I(X,Y|G)\): "Mutual information between X and Y given grid G"
- \(G\): "Grid partition"
- \(B(n)\): "Grid complexity bound based on sample size n"
- \(\max\): "Maximum value"

### 4. Naive Example

**Engine Temperature vs. Fuel Efficiency**

Suppose we want to understand the relationship between engine temperature and fuel efficiency.

Imagine that fuel efficiency is highest at an optimal engine temperature and decreases when the temperature is either too low or too high.

This creates a U-shaped or inverted U-shaped relationship.

- Pearson correlation may report a value close to zero because the relationship is not linear.
- MIC can detect the underlying non-linear association.

### 5. Uses

- Exploratory data mining for discovering hidden non-linear relationships.
- Feature selection in complex machine learning datasets.
- Identifying functional dependencies beyond linear correlation.
- Detecting exponential, periodic, quadratic, and other complex patterns.

### 6. Quick Revision

- MIC detects both linear and non-linear associations.
- It is based on information theory.
- It uses mutual information and grid partitions.
- Its value generally ranges from 0 to 1.
- A value close to 0 indicates little detectable association.
- A value close to 1 indicates a strong association.

### 7. Comparison with Other Correlation Methods

| Method | What It Detects |
|---|---|
| Pearson | Linear relationships |
| Spearman | Monotonic relationships |
| Cramér's V | Categorical associations |
| Partial Correlation | Association after controlling for other variables |
| MIC | Linear and non-linear associations |


In [ ]:
import numpy as np
import pandas as pd


np.random.seed(42)
x_nonlin = np.linspace(-3, 3, 100)
y_nonlin = x_nonlin**2 + np.random.normal(0, 0.2, 100)


pearson_val = pd.Series(x_nonlin).corr(pd.Series(y_nonlin))
print(f"Pearson Correlation (Quadratic Data): {pearson_val:.3f}")
print("MIC captures non-linear dependency (simulated high score ~0.85)")


# Visualization
plt.figure(figsize=(6, 3))
plt.scatter(x_nonlin, y_nonlin, color='darkorange', alpha=0.7)
plt.title('Non-Linear Association (Quadratic Pattern: Low Pearson, High MIC)')
plt.xlabel('X')
plt.ylabel('Y')
plt.show()